EDA

In [ ]:
def evaluate_model(name, X_sub, y_sub):
    model = GradientBoostingRegressor(random_state=42)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_sub, y_sub, scoring='r2', cv=kf)
    return {
        "Dataset": name,
        "Mean R2": np.mean(scores),
        "Std R2": np.std(scores)
    }

# Evaluasi semua
results = []
for name, (X_sub, y_sub) in data_uji.items():
    result = evaluate_model(name, X_sub, y_sub)
    results.append(result)
df_results = pd.DataFrame(results)
df_results["N Data"] = df_results["Dataset"].str.extract(r"(\d+)").astype(float)

# Urutkan
df_results = df_results.sort_values("N Data")

# Plot
plt.figure(figsize=(10, 6))
plt.errorbar(df_results["N Data"], df_results["Mean R2"], yerr=df_results["Std R2"], fmt='-o')
plt.title("Kinerja Model vs Jumlah Data (Cross-Validated R²)")
plt.xlabel("Jumlah Data")
plt.ylabel("Mean R² Score (+/- Std)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [111]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [112]:
df = pd.read_csv('../data/dataset_clean_robust.csv')

In [113]:
X = df.drop(columns=['index_k'])  # Semua kolom kecuali target
y = df['index_k']  # Kolom target

In [114]:
# Membagi skor kebahagiaan menjadi kategori
bins = [0, 33, 66, 100]
labels = ['Rendah', 'Sedang', 'Tinggi']
y_binned = pd.cut(y, bins=bins, labels=labels)

# Stratified Sampling berdasarkan kategori
from sklearn.model_selection import StratifiedShuffleSplit
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in splitter.split(X, y_binned):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


In [115]:
from sklearn.utils import resample
X_train_resampled, y_train_resampled = resample(X_train, y_train, n_samples=len(X_train), random_state=42)

In [116]:
import numpy as np

# Hitung Q1, Q3, dan IQR
Q1 = y_train_resampled.quantile(0.25)
Q3 = y_train_resampled.quantile(0.75)
IQR = Q3 - Q1

# Tentukan batas bawah dan atas
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Temukan nilai ekstrim
extreme_values = y_train_resampled[(y_train_resampled < lower_bound) | (y_train_resampled > upper_bound)]

print("Nilai Ekstrim (IQR Method):")
print(extreme_values)
print("Nilai Ekstrim (IQR Method):")
print(upper_bound)
print("Nilai Ekstrim (IQR Method):")
print(lower_bound)


Nilai Ekstrim (IQR Method):
2051     83.40
64370    83.58
23451    88.90
1926     84.31
50214    84.53
         ...  
50156    85.85
36247    84.28
60231    46.81
25583    84.08
33623    47.33
Name: index_k, Length: 1430, dtype: float64
Nilai Ekstrim (IQR Method):
83.27499999999999
Nilai Ekstrim (IQR Method):
49.47500000000001


In [117]:
from scipy import stats

# Hitung Z-score untuk setiap data
z_scores = np.abs(stats.zscore(y_train_resampled))

# Tentukan batas untuk outlier
outliers = y_train_resampled[z_scores > 3]

print("Nilai Ekstrim (Z-score Method):")
print(outliers)
# Tentukan threshold berdasarkan Z-score
threshold_high = np.percentile(z_scores, 90)  # Ambang batas Z-score untuk persentil ke-90
threshold_low = np.percentile(z_scores, 10)   # Ambang batas Z-score untuk persentil ke-10

print(f"Threshold Z-score Tinggi: {threshold_high}")
print(f"Threshold Z-score Rendah: {threshold_low}")
weights = np.where((z_scores > threshold_high) | (z_scores < threshold_low), 2, 1)
sample_weights = weights

Nilai Ekstrim (Z-score Method):
23451    88.90
45598    43.57
50256    91.38
7404     88.06
45696    44.30
         ...  
56160    93.94
38366    87.53
4833     38.88
41292    88.36
54951    88.13
Name: index_k, Length: 519, dtype: float64
Threshold Z-score Tinggi: 1.6010393228618156
Threshold Z-score Rendah: 0.11092589245574015


In [118]:
from sklearn.ensemble import RandomForestRegressor

# Model regresi
model = RandomForestRegressor(random_state=42, n_jobs=-1)
# Latih model dengan sample weights
model.fit(X_train_resampled, y_train_resampled)
# Prediksi dan evaluasi
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score : {r2:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"MAE      : {mae:.4f}")


R² Score : 0.4492
MSE      : 25.8997
RMSE     : 5.0892
MAE      : 3.8749


In [119]:
from sklearn.ensemble import RandomForestRegressor

# Model regresi
model_b = RandomForestRegressor(random_state=42, n_jobs=-1)
# Latih model dengan sample weights
model_b.fit(X_train, y_train)
# Prediksi dan evaluasi
y_pred_b = model_b.predict(X_test)
r2_b = r2_score(y_test, y_pred)
mse_b = mean_squared_error(y_test, y_pred)
rmse_b = np.sqrt(mse_b)
mae_b = mean_absolute_error(y_test, y_pred)

print(f"R² Score : {r2_b:.4f}")
print(f"MSE      : {mse_b:.4f}")
print(f"RMSE     : {rmse_b:.4f}")
print(f"MAE      : {mae_b:.4f}")


R² Score : 0.4492
MSE      : 25.8997
RMSE     : 5.0892
MAE      : 3.8749
